In [10]:
import pandas as pd

url = "https://www.data.gouv.fr/api/1/datasets/r/bcd7d584-89c3-46b6-a164-4582eb2d43b5"
df = pd.read_csv(url, sep=";", encoding="utf-8")

print(df.shape)
df.head()

(3237, 8)


,librairie,assortiment,label,Commune,code_commune,Code_postal,Departement,Region
0,BLANCHE NEIGE,Généraliste,NaN,AMBERIEU EN BUGEY,01004,1500,AIN,AUVERGNE RHONE ALPES
1,ENTRE PARENTHESES,Généraliste,NaN,BELLEY,01034,1300,AIN,AUVERGNE RHONE ALPES
2,LIBRAIRIE ZANNINI,Généraliste,NaN,BOURG EN BRESSE,01053,1000,AIN,AUVERGNE RHONE ALPES
3,MONTBARBON,Généraliste,LIR,BOURG EN BRESSE,01053,1000,AIN,AUVERGNE RHONE ALPES
4,ZANNINI,Généraliste,NaN,BOURG EN BRESSE,01053,1000,AIN,AUVERGNE RHONE ALPES


In [13]:
df = df.rename(columns={"code_commune ": "code_commune"})

df = df[["librairie", "assortiment", "Commune", "code_commune"]].copy()

In [20]:
import requests

# Récupération de toutes les communes avec leur centroïde (lat/lon)
url_communes = (
    "https://geo.api.gouv.fr/communes"
    "?fields=code,nom,centre,codesPostaux"
    "&format=json"
    "&geometry=centre"
)

communes = requests.get(url_communes, timeout=60).json()
df_communes = pd.DataFrame(communes)

# Coordonnées depuis "centre.coordinates" -> [lon, lat]
df_communes["lon"] = df_communes["centre"].apply(
    lambda x: x.get("coordinates", [None, None])[0] if isinstance(x, dict) else None
)
df_communes["lat"] = df_communes["centre"].apply(
    lambda x: x.get("coordinates", [None, None])[1] if isinstance(x, dict) else None
)

df_communes = df_communes[["code", "nom", "lon", "lat"]].rename(
    columns={"code": "code_commune", "nom": "nom_commune_ref"}
)

# Merge sur code INSEE commune
df_map = df.merge(df_communes, on="code_commune", how="left")

print("Lignes:", len(df_map))
print("Coordonnées manquantes:", df_map[["lat", "lon"]].isna().any(axis=1).sum())
df_map.head()

Lignes: 3237
Coordonnées manquantes: 0


,librairie,assortiment,Commune,code_commune,nom_commune_ref,lon,lat
0,BLANCHE NEIGE,Généraliste,AMBERIEU EN BUGEY,01004,Ambérieu-en-Bugey,5.3706,45.9575
1,ENTRE PARENTHESES,Généraliste,BELLEY,01034,Belley,5.6927,45.7430
2,LIBRAIRIE ZANNINI,Généraliste,BOURG EN BRESSE,01053,Bourg-en-Bresse,5.2469,46.2027
3,MONTBARBON,Généraliste,BOURG EN BRESSE,01053,Bourg-en-Bresse,5.2469,46.2027
4,ZANNINI,Généraliste,BOURG EN BRESSE,01053,Bourg-en-Bresse,5.2469,46.2027


In [21]:
df_map.columns

Index(['librairie', 'assortiment', 'Commune', 'code_commune',
       'nom_commune_ref', 'lon', 'lat'],
      dtype='str')

In [23]:
nb_librairies = (
    df_map
    .groupby(["code_commune", "lon", "lat", "Commune"], as_index=False)
    .size()
    .rename(columns={"size": "nb_librairies"})
)

In [24]:
nb_librairies

,code_commune,lon,lat,Commune,nb_librairies
0,01004,5.3706,45.9575,AMBERIEU EN BUGEY,1
1,01033,5.7944,46.1234,VALSERHONE,2
2,01034,5.6927,45.7430,BELLEY,1
3,01053,5.2469,46.2027,BOURG EN BRESSE,3
4,01093,4.9549,46.1264,CHATILLON SUR CHALARONNE,2
...,...,...,...,...,...
1381,97502,-56.1805,46.7880,Saint-Pierre-et-Miquelon,1
1382,97606,45.1428,-12.9179,CHIRONGUI,1
1383,97611,45.1964,-12.7875,MAMOUDZOU,2
1384,97701,-62.8601,17.9204,Saint-Barthélémy,2
